# 01 · Cell metadata from CELLxGENE Census
**Week 2.** Fetch metadata only (no expression yet; fast): one row per cell from healthy human blood.
Then choose 4–6 datasets from different studies with plenty of cells.

In [ ]:
import cellxgene_census
import pandas as pd

CENSUS_VERSION = "stable"   # write down the actual version printed below in notes/log.md

census = cellxgene_census.open_soma(census_version=CENSUS_VERSION)
print(census["census_info"]["summary"].read().concat().to_pandas())

In [ ]:
obs = cellxgene_census.get_obs(
    census, "homo_sapiens",
    value_filter=(
        "tissue_general == 'blood' and disease == 'normal' "
        "and is_primary_data == True"
    ),
    column_names=["dataset_id", "donor_id", "assay", "tissue", "suspension_type",
                  "cell_type", "cell_type_ontology_term_id"],
)
obs.shape

In [ ]:
# Which studies do the datasets come from?
datasets = census["census_info"]["datasets"].read().concat().to_pandas()
per_ds = (obs.groupby("dataset_id")
             .agg(n_cells=("cell_type", "size"), n_labels=("cell_type", "nunique"),
                  assays=("assay", lambda s: ", ".join(sorted(s.unique()))))
             .join(datasets.set_index("dataset_id")[["collection_name", "dataset_title", "collection_doi"]])
             .sort_values("n_cells", ascending=False))
per_ds.head(30)

**Your choice:** pick 4–6 datasets from *different collections* (labs), 10x assays, >20,000 cells.
Record them, and why you picked them, in `notes/datasets.md`.

In [ ]:
CHOSEN = [
    # "dataset_id_1",
]
obs.to_parquet("../data/processed/blood_obs.parquet")
census.close()